In [6]:
import os
os.chdir("D:/rag-project")  # 切到项目根目录

%load_ext autoreload   
%autoreload 2           
# 启用 autoreload 扩展
# 模式 2:每次执行 cell 前自动重载所有 import 的模块

import sys
sys.path.append("D:/rag-project")

from config import load_config, configure_settings
from service import RagService

config = load_config()
configure_settings(config)
service = RagService(config)
index = service.components.index
print("✓ 加载完成")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Building retriever with mode: multi_query
Building reranker with strategy: capped
✓ 加载完成


In [39]:
# 所有 nodes 的总数
all_nodes = list(index.docstore.docs.values())
print(f"总 chunks: {len(all_nodes)}")

# 第一个 node 长什么样
n = all_nodes[0]
print(f"type: {type(n).__name__}")
print(f"text: {n.text[:80]}")
print(f"metadata keys: {list(n.metadata.keys())}")
print(f"metadata: {n.metadata}")

总 chunks: 6569
type: TextNode
text: Provided proper attribution is provided, Google hereby grants permission to
repr
metadata keys: ['chunk_index', 'content_type', 'source_file', 'total_chunks', 'position']
metadata: {'chunk_index': 0, 'content_type': 'main', 'source_file': 'Attention_is_all_you_need.txt', 'total_chunks': 59, 'position': '开头'}


In [40]:
庄子_nodes = [n for n in all_nodes if "庄子" in n.metadata.get("source_file", "")]
print(f"庄子 chunks: {len(庄子_nodes)}")

# 统计 content_type 分布
from collections import Counter
types = Counter(n.metadata.get("content_type", "未标") for n in 庄子_nodes)
print(f"分布: {types}")

庄子 chunks: 822
分布: Counter({'main': 773, 'auxiliary': 49})


In [41]:
# 找出庄子.txt 里所有提到"题解"的 chunk
zhuangzi_nodes = [n for n in all_nodes if "庄子" in n.metadata.get("source_file", "")]

# 按 chunk_index 排序,看前 20 个
zhuangzi_nodes.sort(key=lambda n: n.metadata.get("chunk_index", 0))

for n in zhuangzi_nodes:
    if "题解" in n.text[:500]:
        idx = n.metadata.get("chunk_index")
        ct = n.metadata.get("content_type")
        print(f"题解出现在 idx={idx},content_type={ct}")
        print(f"前 200 字:{n.text[:200]}")
        print("---")



题解出现在 idx=75,content_type=auxiliary
前 200 字:二〇一〇年一月
### 【题解】
首篇以“逍遥游”三字名篇，隋唐陆德明认为是取其“闲放不拘，怡适自得”（《经典释文》）
之义。这一解说是正确的。按“逍遥”一词，早在《诗经·郑风·清人》中就已出现，与“翱翔”
同义。而在《楚辞》中尤为多见，如“聊逍遥以相羊”（《离骚》）、“聊逍遥兮容与”（《湘
君》）、“聊仿佯而逍遥”（《远游》）。《庄子》与《楚辞》同为南方文学，故文中亦常用此
词。如本篇中的“彷徨
---
题解出现在 idx=76,content_type=auxiliary
前 200 字:文章先写大鹏凭风南飞，以寓万物皆“有所待”之意。但唯恐他人不信，所以随即引《齐
谐》作为证明，又通过借野马、尘埃、大舟喻大鹏，借水与生物之息喻大风，然后再通
过蜩、学鸠、朝菌、蟪蛄、冥灵、大椿、彭祖、众人与汤之问棘来反复申述此意。接着
以“此小大之辩也”稍作收束，暗示凡此种种，虽有大小之别，寿夭之殊，然其“有所待”，
则皆无例外。但文复生文，喻中夹喻，波兴云委，莫测涯涘，行文至此并未点明正意。
继
---
题解出现在 idx=78,content_type=auxiliary
前 200 字:此篇行文，先以“丧我”发端，暗示物论纷纭不齐，皆由执“我”之见所致，所以要齐而同
之，非先忘“我”不可。接着紧承“丧我”而忽以“三籁”致问，但却又随即撇开“人籁”、“天籁”，
而独将“地籁”铺叙描写一番，为下文穷尽种种人情世态作出铺垫。然后迂回推进，由种
种不齐的人情，逐步导出“是非”二字。于是再深一层，进一步追究产生是非的根源——
“成心”。至此，行文似乎已断。但文章却以“言非吹也”一句，遥接“
---
题解出现在 idx=79,content_type=auxiliary
前 200 字:旧时多以“生主”二字连读，而解为“真君”、“真宰”、“真性”等，似与庄子原意不合。所谓
“养生主”，即“养生之宗旨”。正有如王先谦所云：“顺事而不滞于物，冥情而不撄于天，
此庄子养生之宗主也。”（《庄子集解》）也就是说，循乎天理，依乎自然，处于至虚，游
于无有，完全取消主客对立，使精神不为外物所伤，最后达到享尽天年的目的，乃是《养
生主》一文的宗旨。
全篇是以“缘督以为经”为纲，通过三则寓言故事来
---
题解出现

In [42]:
for n in zhuangzi_nodes:
    if n.metadata.get("chunk_index") == 75:
        print(f"content_type: {n.metadata.get('content_type')}")
        print(f"前 200 字: {n.text[:200]}")

content_type: auxiliary
前 200 字: 二〇一〇年一月
### 【题解】
首篇以“逍遥游”三字名篇，隋唐陆德明认为是取其“闲放不拘，怡适自得”（《经典释文》）
之义。这一解说是正确的。按“逍遥”一词，早在《诗经·郑风·清人》中就已出现，与“翱翔”
同义。而在《楚辞》中尤为多见，如“聊逍遥以相羊”（《离骚》）、“聊逍遥兮容与”（《湘
君》）、“聊仿佯而逍遥”（《远游》）。《庄子》与《楚辞》同为南方文学，故文中亦常用此
词。如本篇中的“彷徨


In [11]:
# 找 content_type 发生变化的那些 chunk,看切换是不是发生在合理位置
prev_type = None
for n in sorted_nodes:
    ct = n.metadata.get("content_type")
    if ct != prev_type:
        idx = n.metadata.get("chunk_index")
        preview = n.text[:80].replace("\n", " ")
        print(f"→ 切换到 [{ct}] @ chunk_index={idx}")
        print(f"   {preview}")
        print()
    prev_type = ct

→ 切换到 [auxiliary] @ chunk_index=0
   # 前言 ## 庄子其人 关于庄子的历史记载颇少，其生前默默无闻，死后也长时间少有人问津，以致家世渊源、 师承关系、生卒年月均不甚明了。在战国时期的人之中，除了

→ 切换到 [main] @ chunk_index=45
   ## 庄子的艺术特色 寓言、重言、卮言的运用是《庄子》一书最重要的艺术特色。庄子在《寓言》中曾自叙 其著述特点为：“寓言十九，重言十七，卮言日出，和以天倪。”在

→ 切换到 [auxiliary] @ chunk_index=90
   ### 【题解】 本篇运用《齐物论》的观点，极力论证万物大小、是非的无限相对性和人生贵贱、荣辱 的极端无常性，旨在要人息伪还真，顺应自然，不为追求名位、富贵等而

→ 切换到 [main] @ chunk_index=138
   ### 【正文】 老聃死，秦失吊之，三号而出。弟子曰：“非夫子之友邪？”曰：“然。”“然则吊焉若此可 乎？”曰：“然。始也吾以为其人也，而今非也。向吾入而吊焉，



In [ ]:
'''
 * @Author       : MatthewZhang
 * @Date         : 2026-04-10 11:46:18
 * @Description  : 
'''
from Chunker import propagate_content_type, Chunk

# 构造 5 个假 chunks 模拟庄子结构
fake_chunks = [
    Chunk(text="# 前言\n关于庄子的背景..."),
    Chunk(text="庄子的生平记载很少..."),
    Chunk(text="## 【题解】\n逍遥游一词的含义..."),
    Chunk(text="# 逍遥游\n北冥有鱼,其名为鲲..."),
    Chunk(text="鲲之大,不知其几千里也..."),
]

result = propagate_content_type(fake_chunks)
for c in result:
    ct = c.metadata.get("content_type", "?")
    preview = c.text[:40].replace("\n", " ")
    print(f"[{ct:10}] {preview}")

[auxiliary ] # 前言 关于庄子的背景...
[auxiliary ] 庄子的生平记载很少...
[auxiliary ] ## 【题解】 逍遥游一词的含义...
[main      ] # 逍遥游 北冥有鱼,其名为鲲...
[main      ] 鲲之大,不知其几千里也...


In [5]:
def my_sum(a: int, b: int) -> int:
    return a + b

res = my_sum(2,3)
print(res)

5


In [ ]:
from typing import List, Optional

from llama_index.core.schema import NodeWithScore, TextNode, QueryBundle
from llama_index.core.postprocessor.types import BaseNodePostprocessor

class Auxiliary_downweight_postprocessor(BaseNodePostprocessor):
    weight: float = 0.5

    # 输入: 只需要nodes->List[NodeWithScore], 不需要query_bundle
    # 输出: 新的处理过的List[NodeWithScore]
    # 过程: 把List[NodeWithScore]中的score分数降权
    def _postprocess_nodes(self, nodes: List[NodeWithScore], query_bundle: Optional[QueryBundle] = None) -> List[NodeWithScore]:
        for n in nodes:
            if n.node.metadata.get("content_type") == "auxiliary":
                n.score = (n.score or 0) * self.weight
        nodes.sort(key=lambda x: (x.score or 0), reverse=True)
        return nodes


In [16]:
processor = Auxiliary_downweight_postprocessor(weight=0.5)
print(processor.weight)

0.5


In [18]:
nodes = [
    NodeWithScore(node=TextNode(text="正文A", metadata={"content_type": "main"}), score=0.8),
    NodeWithScore(node=TextNode(text="前言B", metadata={"content_type": "auxiliary"}), score=0.9),
    NodeWithScore(node=TextNode(text="老数据C", metadata={}), score=0.7),
]
result = processor.postprocess_nodes(nodes)
for r in result:    # r是NodeWithScore类型
    print(r.text, r.score)
a = Auxiliary_downweight_postprocessor(weight=0.5)
print(type(a))
print(a)

正文A 0.8
老数据C 0.7
前言B 0.45
<class '__main__.Auxiliary_downweight_postprocessor'>
callback_manager=<llama_index.core.callbacks.base.CallbackManager object at 0x0000021F12413EF0> weight=0.5


In [19]:
# 造 5 个 nodes,模拟 reranker 输出后的样子
nodes = [
    NodeWithScore(node=TextNode(text="前言A", metadata={"content_type": "auxiliary"}), score=0.95),
    NodeWithScore(node=TextNode(text="前言B", metadata={"content_type": "auxiliary"}), score=0.92),
    NodeWithScore(node=TextNode(text="正文C", metadata={"content_type": "main"}), score=0.85),
    NodeWithScore(node=TextNode(text="正文D", metadata={"content_type": "main"}), score=0.75),
    NodeWithScore(node=TextNode(text="前言E", metadata={"content_type": "auxiliary"}), score=0.70),
]

processor = Auxiliary_downweight_postprocessor(weight=0.5)
result = processor.postprocess_nodes(nodes)

for r in result:
    print(f"{r.text}  score={r.score}")

正文C  score=0.85
正文D  score=0.75
前言A  score=0.475
前言B  score=0.46
前言E  score=0.35


In [5]:
from llama_index.core.retrievers import VectorIndexRetriever
from router import route_query

retriever = VectorIndexRetriever(index=index, similarity_top_k=10)

question = "尼采和帕斯卡尔对信仰的态度有何不同？"

intent = 'multi_doc'

components = service.components

result = route_query(question, intent, components)
print(result)
assert "answer" in result
assert "sources" in result
assert isinstance(result["sources"], list)



  → intent: multi_doc, question: 尼采和帕斯卡尔对信仰的态度有何不同？
{'answer': '根据参考资料，尼采与帕斯卡尔对信仰的态度存在根本差异：\n\n**尼采的态度**：  \n- 批判传统信仰，尤其是基督教信仰。他认为当代人（“现实者”）缺乏真正的信仰，只是过去信仰的破碎残余，无法创造新价值。  \n- 提倡“忠实于大地”的“超人”信仰，即摆脱对虚构上帝的依赖，以尘世标准为准绳，通过权力意志和永恒轮回思想确立新价值。  \n- 强调信仰应源于创造者的“真实梦想和星象”，而非盲从或迷信。\n\n**帕斯卡尔的态度**：  \n- 坚定信仰基督教（冉森派），将信仰视为人生目的，追求通过神恩实现自我完善和“神圣性”。  \n- 区分科学与神学，认为信仰与理性并不矛盾，但属于不同范畴；信仰不能通过理性证明，而需依靠神恩启示。  \n- 信仰带有悲观宿命论色彩，认为只有被上帝选中者才能得救，但信仰能为人提供精神支柱，应对世俗世界的迷茫。\n\n**核心区别**：  \n尼采反对传统信仰，主张以人的力量创造新价值；帕斯卡尔则虔诚皈依基督教，视信仰为超越世俗和精神领域的终极归宿。两者分别代表了“颠覆信仰”与“委身信仰”的不同路径。', 'sources': [{'score': 0.0164, 'source_file': '查拉图斯特拉如是说 (尼采经典著作) (尼采 [尼采]) (Z-Library).txt', 'heading': '', 'position': '中间', 'text': '这一点，是的，就是这一点，乃我内心的痛苦：我既不能忍受你们赤裸，又不能忍受你们穿着，你们这些当代人呵！\n未来的一切阴森可怕，以及向来使迷路之鸟战栗的东西，委实都比你们的“现实”更隐秘和更亲切。\n因为你们说：“我们完全是现实的，毫无信仰和迷信”：你们就这样自鸣得意——呵，也还没有自夸的胸腔！\n是的，你们这些斑杂多彩者呵，你们如何能够 信仰！——你们乃是一切向来被信仰的东西的图画！\n你们乃是信仰本身的变化不定的反驳，以及对一切思想的肢解。不可信者 ：我 这样叫你们，你们这些现实者呵！\n所有时代都在你们的精神里彼此喋喋不休；所有时代的梦想和闲言都要比你们的清醒更现实！\n你们是不会生育者：因此 你们缺...'}, {'score': 0.

In [7]:
import time
from llama_index.core.retrievers import VectorIndexRetriever
from router import route_query

retriever = VectorIndexRetriever(index=index, similarity_top_k=10)

question = "道德经的'无为'和佛教的'空'有什么关系？"

intent = 'multi_doc'

components = service.components

result = route_query(question, intent, components)

  → intent: multi_doc, question: 道德经的'无为'和佛教的'空'有什么关系？
并发耗时: 0.7581739999986894 s
并发耗时: 4.599998646881431e-06 s
并发耗时: 1.3999997463542968e-06 s


In [4]:
print(type(query_engine))
print(query_engine.__class__.__mro__)

<class 'llama_index.core.query_engine.retriever_query_engine.RetrieverQueryEngine'>
(<class 'llama_index.core.query_engine.retriever_query_engine.RetrieverQueryEngine'>, <class 'llama_index.core.base.base_query_engine.BaseQueryEngine'>, <class 'llama_index.core.prompts.mixin.PromptMixin'>, <class 'llama_index_instrumentation.DispatcherSpanMixin'>, <class 'abc.ABC'>, <class 'object'>)
